# MoveNet + TensorRT + GRU Fall Detection Pipeline

**Target Device**: Jetson Nano | JetPack 4.6.x | Python 3.6 | CUDA 10.2 | TensorRT 8.2

## Workflow

```
Step 0: Environment check & setup
Step 1: Clone movenet.pytorch, prepare pretrained weights
Step 2: PTH -> ONNX conversion (done on PC, copy .onnx to Nano)
Step 3: ONNX -> TensorRT Engine (FP16) — must run on Nano
Step 4: TensorRT real-time pose inference (webcam / video)
Step 5: Collect keypoint data -> train GRU fall detector
Step 6: Full MoveNet + GRU pipeline running in real-time
```

---

## Step 0: Environment Check

In [ ]:
import sys
import os

print("=" * 50)
print("Environment Info")
print("=" * 50)
print("Python:  {}".format(sys.version))

try:
    import torch
    print("PyTorch: {}".format(torch.__version__))
    print("CUDA:    {} (available={})".format(
        torch.version.cuda, torch.cuda.is_available()))
    if torch.cuda.is_available():
        print("GPU:     {}".format(torch.cuda.get_device_name(0)))
        print("Memory:  {:.0f} MB".format(
            torch.cuda.get_device_properties(0).total_mem / 1024**2))
except ImportError:
    print("[ERROR] PyTorch not installed!")
    print("  wget https://nvidia.box.com/shared/static/h1z9sw4bb1ybi0rm3tu8qdj8hs05ljbm.whl -O torch-1.9.0-cp36-cp36m-linux_aarch64.whl")
    print("  pip3 install torch-1.9.0-cp36-cp36m-linux_aarch64.whl")

try:
    import tensorrt as trt
    print("TensorRT: {}".format(trt.__version__))
except ImportError:
    print("[ERROR] TensorRT not installed! (should be pre-installed with JetPack)")
    print("  sudo apt-get install tensorrt python3-libnvinfer-dev")

try:
    import onnx
    print("ONNX:    {}".format(onnx.__version__))
except ImportError:
    print("ONNX:    not installed -> pip3 install onnx==1.12.0")

try:
    import cv2
    print("OpenCV:  {}".format(cv2.__version__))
except ImportError:
    print("OpenCV:  not installed")

try:
    import pycuda.driver
    print("PyCUDA:  OK")
except ImportError:
    print("PyCUDA:  not installed -> pip3 install pycuda")

print("=" * 50)

### 0.1 Install Missing Dependencies (run as needed)

In [ ]:
# ===== Uncomment lines you need =====

# !pip3 install numpy==1.19.5
# !pip3 install onnx==1.12.0 protobuf==3.20.3
# !pip3 install onnx-simplifier==0.4.17
# !pip3 install onnxruntime==1.11.0
# !pip3 install pycuda

### 0.2 Set Jetson Nano to Max Performance + Enable Swap

In [ ]:
# Max performance mode (10W, all 4 cores)
!sudo nvpmodel -m 0
!sudo jetson_clocks

# Add swap (needed for TRT engine build)
import os
if not os.path.exists('/swapfile'):
    print("Creating 4GB swap...")
    !sudo fallocate -l 4G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    !sudo swapon /swapfile
else:
    print("Swap already exists")

!free -h | grep -i swap

---
## Step 1 & 2: Prepare ONNX Model

The ONNX conversion should be done on your **PC** using `pth2onnx.py`:

```bash
# On PC:
cd movenet.pytorch
python pth2onnx.py
# -> generates output/pose_train.onnx and output/pose_test.onnx

# Copy to Jetson Nano:
scp output/pose_test.onnx nano@<NANO_IP>:~/
```

Set the ONNX file path below:

In [ ]:
# ============================================================
#  *** EDIT HERE: point to your ONNX file on Nano ***
# ============================================================
ONNX_PATH     = "pose_test.onnx"           # <- your .onnx file
ENGINE_PATH   = "pose_fp16.engine"          # <- TRT engine output
IMG_SIZE      = 192

# Check file
if os.path.exists(ONNX_PATH):
    size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    print("[OK] ONNX file: {} ({:.1f} MB)".format(ONNX_PATH, size_mb))
else:
    print("[ERROR] ONNX file not found: {}".format(ONNX_PATH))
    print("  Please copy it from your PC first!")

---
## Step 3: ONNX -> TensorRT Engine

**This step MUST run on Jetson Nano!** Engine files are architecture-specific.

Two options:
- **Option A**: Use `trtexec` command-line tool (simpler, recommended)
- **Option B**: Use Python API below (TRT 8.2 compatible)

In [ ]:
# ===== Option A: trtexec (recommended, simpler and more stable) =====
# Uncomment the line below to use trtexec directly:

# !/usr/src/tensorrt/bin/trtexec --onnx=pose_test.onnx --saveEngine=pose_fp16.engine --fp16 --workspace=1024

In [ ]:
# ===== Option B: Python API (TRT 8.2 compatible) =====
import tensorrt as trt
import time

TRT_LOGGER = trt.Logger(trt.Logger.INFO)
USE_FP16 = True       # <- FP16 recommended for Jetson Nano
WORKSPACE_GB = 1      # <- Nano has limited RAM, 1GB is enough

print("[INFO] TensorRT {}".format(trt.__version__))
print("[INFO] Building engine (may take 5-15 minutes)...")
print("[INFO] If killed by OOM, make sure swap is enabled")

# 1. Create builder / network / parser
builder = trt.Builder(TRT_LOGGER)
network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
network = builder.create_network(network_flags)
parser = trt.OnnxParser(network, TRT_LOGGER)

# 2. Parse ONNX
with open(ONNX_PATH, 'rb') as f:
    if not parser.parse(f.read()):
        for i in range(parser.num_errors):
            print("[ERROR] {}".format(parser.get_error(i)))
        raise RuntimeError("ONNX parsing failed")

print("[OK] ONNX parsed successfully")
for i in range(network.num_inputs):
    inp = network.get_input(i)
    print("  Input:  {} -> {}".format(inp.name, inp.shape))
for i in range(network.num_outputs):
    out = network.get_output(i)
    print("  Output: {} -> {}".format(out.name, out.shape))

# 3. Configure (TRT 8.2 API)
config = builder.create_builder_config()
config.max_workspace_size = WORKSPACE_GB * (1 << 30)   # <- TRT 8.2 uses this

if USE_FP16 and builder.platform_has_fast_fp16:
    config.set_flag(trt.BuilderFlag.FP16)
    print("[OK] FP16 enabled")

# 4. Build engine (TRT 8.2: build_engine)
t0 = time.time()

engine = None
try:
    # Try new API first (TRT 8.5+)
    serialized = builder.build_serialized_network(network, config)
    if serialized:
        with open(ENGINE_PATH, 'wb') as f:
            f.write(serialized)
        engine = True
except AttributeError:
    pass

if engine is None:
    # TRT 8.2 fallback
    engine = builder.build_engine(network, config)
    if engine is None:
        raise RuntimeError("Engine build failed! Likely OOM - check swap")
    with open(ENGINE_PATH, 'wb') as f:
        f.write(engine.serialize())

elapsed = time.time() - t0
size_mb = os.path.getsize(ENGINE_PATH) / (1024 * 1024)

print("\n" + "=" * 50)
print("[DONE] Engine: {} ({:.1f} MB)".format(ENGINE_PATH, size_mb))
print("       Build time: {:.0f} seconds".format(elapsed))
print("=" * 50)

---
## Step 4: TensorRT Real-time Pose Inference

Run 17-keypoint detection using the TensorRT engine.

In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import numpy as np
import cv2
import time
from collections import deque

# =====================================================================
# TRT 8.2 Inference Engine
# =====================================================================
class TRTInferencer(object):
    """
    TensorRT 8.2 inference.
    API: engine[i], get_binding_shape, execute_async_v2
    """
    def __init__(self, engine_path):
        self.logger = trt.Logger(trt.Logger.WARNING)
        print("[INFO] Loading engine: {}".format(engine_path))

        with open(engine_path, 'rb') as f:
            runtime = trt.Runtime(self.logger)
            self.engine = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()

        self.bindings = []
        self.binding_addrs = []
        self.input_idx = []
        self.output_idx = []
        self.output_names = []

        for i in range(self.engine.num_bindings):
            name = self.engine[i]
            shape = tuple(self.engine.get_binding_shape(i))
            dtype = trt.nptype(self.engine.get_binding_dtype(i))
            shape = tuple(max(1, s) for s in shape)

            host_mem = np.empty(shape, dtype=dtype)
            device_mem = cuda.mem_alloc(host_mem.nbytes)

            self.bindings.append({
                'name': name, 'shape': shape, 'dtype': dtype,
                'host': host_mem, 'device': device_mem,
            })
            self.binding_addrs.append(int(device_mem))

            if self.engine.binding_is_input(i):
                self.input_idx.append(i)
                print("  Input  [{}]: {} {}".format(i, name, shape))
            else:
                self.output_idx.append(i)
                self.output_names.append(name)
                print("  Output [{}]: {} {}".format(i, name, shape))

        self.stream = cuda.Stream()
        print("[OK] Engine loaded")

    def infer(self, input_data):
        i = self.input_idx[0]
        b = self.bindings[i]
        inp = input_data.astype(b['dtype'])

        cuda.memcpy_htod_async(b['device'], np.ascontiguousarray(inp), self.stream)
        self.context.execute_async_v2(self.binding_addrs, self.stream.handle)

        results = []
        for oi in self.output_idx:
            ob = self.bindings[oi]
            out_shape = self.context.get_binding_shape(oi)
            host_buf = np.empty(out_shape, dtype=ob['dtype'])
            cuda.memcpy_dtoh_async(host_buf, ob['device'], self.stream)
            results.append(host_buf)

        self.stream.synchronize()
        return [r.astype(np.float32) for r in results]


print("[OK] TRTInferencer defined")

In [ ]:
# =====================================================================
# Preprocessing / Postprocessing / Visualization
# =====================================================================

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16),
]

COLORS = [
    (0,255,0),(255,128,0),(0,128,255),(255,0,0),(0,0,255),
    (255,255,0),(0,255,255),(255,0,128),(128,0,255),(255,0,255),
    (0,255,128),(128,255,0),(0,128,128),(128,128,255),
    (255,128,128),(128,255,255),(255,255,128),
]


def preprocess(frame, img_size=192):
    """BGR frame -> (1, 3, H, W) float32 with letterbox"""
    h, w = frame.shape[:2]
    scale = min(float(img_size)/h, float(img_size)/w)
    new_h, new_w = int(h*scale), int(w*scale)
    pad_h = (img_size - new_h) // 2
    pad_w = (img_size - new_w) // 2
    resized = cv2.resize(frame, (new_w, new_h))
    canvas = np.full((img_size, img_size, 3), 128, dtype=np.uint8)
    canvas[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
    blob = canvas.astype(np.float32) / 255.0
    blob = np.transpose(blob, (2,0,1))
    blob = np.expand_dims(blob, 0)
    return blob, scale, pad_w, pad_h


def postprocess_train(outputs, img_w, img_h, img_size=192, scale=1.0, pad_w=0, pad_h=0):
    """4 outputs -> keypoints list"""
    heatmap, center, regs, offsets = outputs[0], outputs[1], outputs[2], outputs[3]
    center_map = center[0, 0]
    cy, cx = np.unravel_index(np.argmax(center_map), center_map.shape)

    keypoints = []
    feat_stride = float(img_size) / heatmap.shape[2]
    for k in range(heatmap.shape[1]):
        rx = float(regs[0, k*2, cy, cx])
        ry = float(regs[0, k*2+1, cy, cx])
        kx = max(0, min(cx + rx, heatmap.shape[3]-1))
        ky = max(0, min(cy + ry, heatmap.shape[2]-1))
        kxi, kyi = int(round(kx)), int(round(ky))
        ox = float(offsets[0, k*2, kyi, kxi])
        oy = float(offsets[0, k*2+1, kyi, kxi])
        x = ((kx + ox) * feat_stride - pad_w) / scale
        y = ((ky + oy) * feat_stride - pad_h) / scale
        score = float(heatmap[0, k, kyi, kxi])
        keypoints.append((x, y, score))
    return keypoints


def postprocess_test(outputs, img_w, img_h, img_size=192, scale=1.0, pad_w=0, pad_h=0):
    """1 output -> keypoints list"""
    kpts = outputs[0]
    if kpts.ndim == 3:
        kpts = kpts[0]
    keypoints = []
    for k in range(kpts.shape[0]):
        x, y = float(kpts[k, 0]), float(kpts[k, 1])
        score = float(kpts[k, 2]) if kpts.shape[1] > 2 else 1.0
        if 0 <= x <= 1 and 0 <= y <= 1:
            x, y = x * img_size, y * img_size
        x = (x - pad_w) / scale
        y = (y - pad_h) / scale
        keypoints.append((x, y, score))
    return keypoints


def draw_keypoints(frame, keypoints, conf=0.3):
    """Draw skeleton and keypoints on frame"""
    h, w = frame.shape[:2]
    for (i, j) in SKELETON:
        if keypoints[i][2] > conf and keypoints[j][2] > conf:
            p1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            p2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(frame, p1, p2, (200,200,200), 2)
    for k, (x, y, s) in enumerate(keypoints):
        if s > conf:
            cv2.circle(frame, (int(x), int(y)), 5, COLORS[k%len(COLORS)], -1)
    return frame


print("[OK] Preprocessing / postprocessing / visualization functions defined")

In [ ]:
# =====================================================================
# Quick test with a dummy image (verify engine works)
# =====================================================================
inferencer = TRTInferencer(ENGINE_PATH)

# Detect output mode
num_outputs = len(inferencer.output_idx)
use_train_post = (num_outputs >= 4)
print("Output count: {} -> using '{}' postprocessing".format(
    num_outputs, "train" if use_train_post else "test"))

# Test with random image
test_img = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
blob, scale, pw, ph = preprocess(test_img, IMG_SIZE)
outputs = inferencer.infer(blob)

print("\nInference outputs:")
for name, out in zip(inferencer.output_names, outputs):
    print("  {}: shape={}, range=[{:.4f}, {:.4f}]".format(
        name, out.shape, out.min(), out.max()))

print("\n[OK] TRT engine inference working!")

In [ ]:
# =====================================================================
# Real-time webcam inference
# =====================================================================
# Change SOURCE: 0=webcam, or path to video file
SOURCE = 0               # <- webcam
# SOURCE = "test.mp4"    # <- video file

SAVE_KEYPOINTS = None    # set to filename to save: "keypoints.json"
MAX_FRAMES = 300         # max frames (prevents notebook from hanging), set None for infinite

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    print("[ERROR] Cannot open video source: {}".format(SOURCE))
else:
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print("[INFO] Source: {} ({}x{})".format(SOURCE, img_w, img_h))

    fps_deque = deque(maxlen=30)
    kpts_list = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()
        blob, scale, pw, ph = preprocess(frame, IMG_SIZE)
        outputs = inferencer.infer(blob)

        if use_train_post:
            kpts = postprocess_train(outputs, img_w, img_h, IMG_SIZE, scale, pw, ph)
        else:
            kpts = postprocess_test(outputs, img_w, img_h, IMG_SIZE, scale, pw, ph)

        dt = time.time() - t0
        fps_deque.append(dt)
        fps = 1.0 / (sum(fps_deque) / len(fps_deque))

        if SAVE_KEYPOINTS:
            kpts_list.append({'frame': frame_idx, 'keypoints': kpts})

        vis = draw_keypoints(frame.copy(), kpts)
        cv2.putText(vis, "FPS: {:.1f}".format(fps),
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        cv2.imshow('MoveNet TRT', vis)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        frame_idx += 1
        if MAX_FRAMES and frame_idx >= MAX_FRAMES:
            break

    cap.release()
    cv2.destroyAllWindows()

    if SAVE_KEYPOINTS and kpts_list:
        import json
        with open(SAVE_KEYPOINTS, 'w') as f:
            json.dump(kpts_list, f)
        print("[OK] Keypoints saved: {} ({} frames)".format(SAVE_KEYPOINTS, len(kpts_list)))

    print("[DONE] {} frames, avg FPS: {:.1f}".format(
        frame_idx, 1.0 / (sum(fps_deque) / len(fps_deque))))

---
## Step 5: GRU Fall Detection Model

### 5.1 Data Preparation

Use Step 4 with `SAVE_KEYPOINTS` to collect keypoint sequences, then manually add labels.

Data format:
```json
{
    "label": 1,
    "frames": [
        {"keypoints": [[x,y,score], ... x17]},
        ...
    ]
}
```

In [ ]:
# Generate sample data (for testing the pipeline)
import json

os.makedirs("data/labeled", exist_ok=True)

for label, name in [(0, "normal"), (1, "fall")]:
    for idx in range(3):
        sample = {"label": label, "frames": []}
        for _ in range(60):
            kpts = []
            for _ in range(17):
                kpts.append([
                    float(np.random.uniform(100, 500)),
                    float(np.random.uniform(100, 500)),
                    float(np.random.uniform(0.5, 1.0)),
                ])
            sample["frames"].append({"keypoints": kpts})

        fname = "data/labeled/{}_{:03d}.json".format(name, idx)
        with open(fname, 'w') as f:
            json.dump(sample, f)

print("[OK] Sample data generated")
!ls data/labeled/

### 5.2 Define GRU Model + Feature Extractor

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class KeypointFeatureExtractor(object):
    """Extract 76-dim features from 17 keypoints"""

    ANGLE_TRIPLETS = [
        (5,7,9),(6,8,10),(11,13,15),(12,14,16),(5,11,13),(6,12,14)
    ]

    def __init__(self):
        self.prev_kpts = None

    def extract(self, keypoints):
        kpts = np.array(keypoints, dtype=np.float32)
        coords, scores = kpts[:, :2], kpts[:, 2]
        features = []

        # Normalized coords (34) + scores (17)
        valid = scores > 0.3
        if np.any(valid):
            cx, cy = coords[valid, 0].mean(), coords[valid, 1].mean()
            s = max(coords[valid, 0].ptp(), coords[valid, 1].ptp(), 1.0)
            norm = (coords - [cx, cy]) / s
        else:
            norm = coords
        features.extend([norm.flatten(), scores])

        # Joint angles (6)
        angles = []
        for a, b, c in self.ANGLE_TRIPLETS:
            if scores[a]>0.3 and scores[b]>0.3 and scores[c]>0.3:
                ba = coords[a] - coords[b]
                bc = coords[c] - coords[b]
                cos = np.clip(np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-8), -1, 1)
                angles.append(np.degrees(np.arccos(cos)) / 180.0)
            else:
                angles.append(0.0)
        features.append(np.array(angles, dtype=np.float32))

        # Body ratio + center of mass height (2)
        sw = np.linalg.norm(coords[5]-coords[6]) if scores[5]>0.3 and scores[6]>0.3 else 0
        bh = 0.0
        if scores[0]>0.3:
            ay = [coords[i,1] for i in [15,16] if scores[i]>0.3]
            if ay: bh = max(ay) - coords[0,1]
        ratio = sw / (abs(bh)+1e-8)
        ch = 0.0
        if scores[11]>0.3 and scores[12]>0.3 and bh>0:
            ch = ((coords[11,1]+coords[12,1])/2) / (abs(bh)+1e-8)
        features.append(np.array([ratio, ch], dtype=np.float32))

        # Inter-frame velocity (17)
        if self.prev_kpts is not None:
            speed = np.linalg.norm(coords - self.prev_kpts[:,:2], axis=1)
        else:
            speed = np.zeros(17, dtype=np.float32)
        features.append(speed)
        self.prev_kpts = kpts.copy()

        return np.concatenate(features).astype(np.float32)

    def reset(self):
        self.prev_kpts = None


class FallDetectionGRU(nn.Module):
    """GRU-based fall detection with attention"""
    def __init__(self, input_dim=76, hidden_dim=64, num_layers=2,
                 num_classes=2, dropout=0.3):
        super(FallDetectionGRU, self).__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers>1 else 0)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes))

    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.gru(x)
        attn = torch.softmax(self.attention(out), dim=1)
        ctx = torch.sum(out * attn, dim=1)
        return self.classifier(ctx)


class FallDataset(Dataset):
    """Load labeled keypoint JSON files for training"""
    def __init__(self, data_dir, seq_len=30, stride=5):
        self.samples = []
        self.feature_dim = 76
        ext = KeypointFeatureExtractor()

        if not os.path.exists(data_dir):
            return
        for fname in sorted(os.listdir(data_dir)):
            if not fname.endswith('.json'):
                continue
            with open(os.path.join(data_dir, fname)) as f:
                data = json.load(f)
            ext.reset()
            feats = [ext.extract(fr['keypoints']) for fr in data['frames']]
            for s in range(0, len(feats)-seq_len+1, stride):
                self.samples.append((np.stack(feats[s:s+seq_len]), data['label']))

        print("[INFO] {} samples loaded".format(len(self.samples)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, label = self.samples[idx]
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(label, dtype=torch.long)


print("[OK] GRU model + dataset defined")
print("  Feature dim: 76")
print("  GRU: input=76, hidden=64, layers=2")

### 5.3 Train GRU

In [ ]:
# =================== Training Config ===================
DATA_DIR    = "data/labeled/"
GRU_SAVE    = "models/fall_gru.pth"
SEQ_LEN     = 30       # sliding window frames (~1s at 30fps)
HIDDEN_DIM  = 64       # small for Nano memory
NUM_LAYERS  = 2
BATCH_SIZE  = 16       # small for Nano memory
EPOCHS      = 30
LR          = 0.001
# =======================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("[INFO] Training on {}".format(device))

dataset = FallDataset(DATA_DIR, seq_len=SEQ_LEN)
if len(dataset) == 0:
    print("[ERROR] No data! Prepare training data first.")
else:
    val_n = max(1, int(len(dataset) * 0.2))
    train_set, val_set = torch.utils.data.random_split(
        dataset, [len(dataset) - val_n, val_n])

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                              shuffle=True, drop_last=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

    gru_model = FallDetectionGRU(
        input_dim=76, hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS
    ).to(device)

    params = sum(p.numel() for p in gru_model.parameters())
    print("[INFO] GRU params: {:,}".format(params))

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(gru_model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0
    for epoch in range(EPOCHS):
        gru_model.train()
        total_loss, correct, total = 0.0, 0, 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            logits = gru_model(bx)
            loss = criterion(logits, by)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gru_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(1) == by).sum().item()
            total += bx.size(0)
        scheduler.step()

        gru_model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                vc += (gru_model(bx).argmax(1) == by).sum().item()
                vt += bx.size(0)
        val_acc = float(vc) / max(vt, 1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print("  Epoch {}/{} | Loss: {:.4f} | TrainAcc: {:.4f} | ValAcc: {:.4f}".format(
                epoch+1, EPOCHS, total_loss/total, float(correct)/total, val_acc))

        if val_acc >= best_acc:
            best_acc = val_acc
            os.makedirs(os.path.dirname(GRU_SAVE), exist_ok=True)
            torch.save({
                'model_state': gru_model.state_dict(),
                'input_dim': 76, 'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_LAYERS, 'seq_len': SEQ_LEN,
                'val_acc': val_acc,
            }, GRU_SAVE)

    print("\n[DONE] Best val_acc: {:.4f}".format(best_acc))
    print("  Model saved: {}".format(GRU_SAVE))

---
## Step 6: Full Pipeline — MoveNet + GRU Real-time Fall Detection

Combine TensorRT pose estimation and GRU fall detection into one real-time pipeline.

In [ ]:
# =================== Pipeline Config ===================
MOVENET_ENGINE = ENGINE_PATH      # TRT engine
GRU_WEIGHTS    = GRU_SAVE         # GRU weights
VIDEO_SOURCE   = 0                # 0=webcam
FALL_THRESHOLD = 0.7              # fall probability threshold
ALARM_FRAMES   = 5                # consecutive frames to trigger alarm
MAX_FRAMES     = 500              # max frames
# =======================================================

# Load MoveNet
pose_engine = TRTInferencer(MOVENET_ENGINE)
num_out = len(pose_engine.output_idx)
use_train = (num_out >= 4)

# Load GRU
gru = None
if os.path.exists(GRU_WEIGHTS):
    ckpt = torch.load(GRU_WEIGHTS, map_location=device)
    gru = FallDetectionGRU(
        input_dim=ckpt['input_dim'],
        hidden_dim=ckpt['hidden_dim'],
        num_layers=ckpt['num_layers'],
    ).to(device)
    gru.load_state_dict(ckpt['model_state'])
    gru.eval()
    seq_len = ckpt.get('seq_len', 30)
    print("[OK] GRU loaded")
else:
    seq_len = 30
    print("[WARN] GRU weights not found, running pose estimation only")

# Feature extractor + sliding window
feat_ext = KeypointFeatureExtractor()
buffer = deque(maxlen=seq_len)
consecutive_fall = 0

print("[OK] Pipeline initialized")

In [ ]:
# =====================================================================
# Run Full Pipeline
# =====================================================================
cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    print("[ERROR] Cannot open: {}".format(VIDEO_SOURCE))
else:
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print("[RUN] Pipeline running on {}x{}".format(img_w, img_h))
    print("      Press 'q' to quit")

    fps_q = deque(maxlen=30)
    frame_idx = 0
    feat_ext.reset()
    buffer.clear()
    consecutive_fall = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()

        # ---- MoveNet inference ----
        blob, scale, pw, ph = preprocess(frame, IMG_SIZE)
        outputs = pose_engine.infer(blob)
        if use_train:
            kpts = postprocess_train(outputs, img_w, img_h, IMG_SIZE, scale, pw, ph)
        else:
            kpts = postprocess_test(outputs, img_w, img_h, IMG_SIZE, scale, pw, ph)

        # ---- GRU fall detection ----
        fall_prob = 0.0
        feat = feat_ext.extract(kpts)
        buffer.append(feat)

        if gru and len(buffer) == seq_len:
            seq = np.stack(list(buffer))
            seq_t = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = torch.softmax(gru(seq_t), dim=1)
                fall_prob = probs[0, 1].item()

        # Alarm logic
        if fall_prob > FALL_THRESHOLD:
            consecutive_fall += 1
        else:
            consecutive_fall = max(0, consecutive_fall - 1)
        is_alarm = consecutive_fall >= ALARM_FRAMES

        # ---- Visualization ----
        dt = time.time() - t0
        fps_q.append(dt)
        fps = 1.0 / (sum(fps_q) / len(fps_q))

        vis = draw_keypoints(frame.copy(), kpts)
        cv2.putText(vis, "FPS: {:.1f}".format(fps),
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        if gru:
            # Fall probability bar
            bw, bh2 = 200, 20
            bx2, by2 = 10, 70
            cv2.rectangle(vis, (bx2,by2), (bx2+bw, by2+bh2), (50,50,50), -1)
            fill = int(bw * fall_prob)
            clr = (0,0,255) if fall_prob > FALL_THRESHOLD else (0,200,0)
            cv2.rectangle(vis, (bx2,by2), (bx2+fill, by2+bh2), clr, -1)
            cv2.putText(vis, "Fall: {:.2f}".format(fall_prob),
                        (bx2+bw+10, by2+15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, clr, 2)
            cv2.putText(vis, "Buf: {}/{}".format(len(buffer), seq_len),
                        (10, by2+bh2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180,180,180), 1)

            if is_alarm:
                cv2.putText(vis, "!! FALL DETECTED !!",
                            (img_w//2-150, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 3)
                cv2.rectangle(vis, (0,0), (img_w-1,img_h-1), (0,0,255), 4)

        cv2.imshow('Fall Detection Pipeline', vis)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        frame_idx += 1
        if MAX_FRAMES and frame_idx >= MAX_FRAMES:
            break

    cap.release()
    cv2.destroyAllWindows()
    print("\n[DONE] {} frames, avg FPS: {:.1f}".format(
        frame_idx, 1.0/(sum(fps_q)/len(fps_q)) if fps_q else 0))

---
## Appendix: Export GRU to ONNX (Optional)

If you want to optimize GRU with TensorRT as well:

In [ ]:
if os.path.exists(GRU_SAVE):
    ckpt = torch.load(GRU_SAVE, map_location='cpu')
    gru_export = FallDetectionGRU(
        input_dim=ckpt['input_dim'],
        hidden_dim=ckpt['hidden_dim'],
        num_layers=ckpt['num_layers'],
    )
    gru_export.load_state_dict(ckpt['model_state'])
    gru_export.eval()

    gru_onnx_path = "models/fall_gru.onnx"
    dummy_seq = torch.randn(1, ckpt['seq_len'], ckpt['input_dim'])

    torch.onnx.export(
        gru_export, dummy_seq, gru_onnx_path,
        input_names=['sequence'], output_names=['prediction'],
        opset_version=11,
    )
    print("[OK] GRU ONNX: {}".format(gru_onnx_path))
    print("  Then convert to TRT engine using trtexec")
else:
    print("[SKIP] No GRU weights found")

---
## Summary

| Step | What | Input | Output |
|------|------|-------|--------|
| 0 | Environment check | - | - |
| 1-2 | PTH->ONNX (on PC) | `.pth` | `.onnx` |
| 3 | ONNX->TRT (on Nano) | `.onnx` | `.engine` |
| 4 | Pose inference | `.engine` + webcam | keypoint visualization |
| 5 | Train GRU | keypoint JSONs | `fall_gru.pth` |
| 6 | Full pipeline | `.engine` + `.pth` + webcam | fall detection |

### Key Notes

- **TRT engine must be built on Nano**, ONNX can be exported on PC
- **FP16 recommended**, Nano's Maxwell GPU does not support INT8
- **If out of memory**: enable swap, close desktop, set workspace to 1GB
- **Performance mode**: `sudo nvpmodel -m 0 && sudo jetson_clocks`
- **Expected FPS**: MoveNet TRT FP16 ~35-50, with GRU ~30-45